<a href="https://colab.research.google.com/github/IdoAbram/Tiny-NMT/blob/dev/notebooks/teacher_nmt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teacher Model Benchmark – English → Spanish

This notebook benchmarks a pretrained MarianMT teacher model.
It measures:
- Translation quality (BLEU, chrF, TER)
- Inference speed
- Model size

The results serve as a fixed baseline for all student models.

## Environment & GPU

In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## Install dependencies

In [7]:
!pip -q install transformers datasets sentencepiece sacrebleu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00


## Load Teacher Model - Helsinki-NLP

In [8]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
teacher = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
teacher.eval()

print("Teacher loaded")

Teacher loaded


## Translation function

In [10]:
def translate_teacher(texts, max_length=128, num_beams=4):
    if isinstance(texts, str):
        texts = [texts]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        outputs = teacher.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams
        )

    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

## Load Dataset (OPUS Books)

In [11]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus_books", "en-es")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93470
    })
})


## Extract Source

In [12]:
def get_src_tgt(split, src_lang="en", tgt_lang="es", limit=None):
    src, tgt = [], []
    for ex in split:
        tr = ex["translation"]
        src.append(tr[src_lang])
        tgt.append(tr[tgt_lang])
        if limit and len(src) >= limit:
            break
    return src, tgt


# use train split, fixed subset
N = 500
train_src, train_ref = get_src_tgt(ds["train"], "en", "es", limit=N)

print("Eval sentences:", len(train_src))

Eval sentences: 500


## Batched Inference Benchmark

In [13]:
import time
from tqdm.auto import tqdm

def batched_translate(texts, batch_size=32, num_beams=4, max_length=128):
    outputs = []
    total_time = 0.0

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        t0 = time.time()
        out = translate_teacher(batch, num_beams=num_beams, max_length=max_length)
        t1 = time.time()

        outputs.extend(out)
        total_time += (t1 - t0)

    return outputs, total_time


preds, total_seconds = batched_translate(
    train_src,
    batch_size=32,
    num_beams=4
)

print("Sentences:", len(preds))
print("Total time (sec):", round(total_seconds, 2))
print("Sentences/sec:", round(len(preds) / total_seconds, 2))

  0%|          | 0/16 [00:00<?, ?it/s]

Sentences: 500
Total time (sec): 35.0
Sentences/sec: 14.29


## Model Size

In [14]:
total_params = sum(p.numel() for p in teacher.parameters())
bytes_per_param = next(teacher.parameters()).element_size()
size_mb = total_params * bytes_per_param / (1024 ** 2)

print(f"Parameters: {total_params:,}")
print(f"Model size: {size_mb:.1f} MB")


Parameters: 77,943,296
Model size: 297.3 MB


## Translation Quality (BLEU / chrF / TER)

In [15]:
from sacrebleu.metrics import BLEU, CHRF, TER

bleu = BLEU()
chrf = CHRF()
ter = TER()

bleu_score = bleu.corpus_score(preds, [train_ref])
chrf_score = chrf.corpus_score(preds, [train_ref])
ter_score = ter.corpus_score(preds, [train_ref])

print("BLEU:", bleu_score)
print("chrF:", chrf_score)
print("TER :", ter_score)

BLEU: BLEU = 27.62 59.3/33.3/21.2/13.9 (BP = 1.000 ratio = 1.004 hyp_len = 14553 ref_len = 14488)
chrF: chrF2 = 53.51
TER : TER = 60.23


## Summary

In [16]:
print("\n=== TEACHER BENCHMARK SUMMARY ===")
print(f"Model: {MODEL_NAME}")
print(f"Sentences: {N}")
print(f"BLEU: {bleu_score.score:.2f}")
print(f"chrF: {chrf_score.score:.2f}")
print(f"TER: {ter_score.score:.2f}")
print(f"Params: {total_params:,}")
print(f"Size: {size_mb:.1f} MB")
print(f"Speed: {len(preds) / total_seconds:.2f} sentences/sec")


=== TEACHER BENCHMARK SUMMARY ===
Model: Helsinki-NLP/opus-mt-en-es
Sentences: 500
BLEU: 27.62
chrF: 53.51
TER: 60.23
Params: 77,943,296
Size: 297.3 MB
Speed: 14.29 sentences/sec
